<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 지식 증류 · 모델 경량화 · 08. ShuffleNet V2

## ShuffleNet V2: Practical Guidelines for Efficient CNN Architecture Design

- **원 논문:** [ShuffleNet V2: Practical Guidelines for Efficient CNN Architecture Design](https://openaccess.thecvf.com/content_ECCV_2018/html/Ningning_Light-weight_CNN_Architecture_ECCV_2018_paper.html)
- **저자 / 발표:** Ningning Ma et al. · ECCV 2018 (2018)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** PyTorch tensor와 autograd, softmax·cross entropy, 선형대수 기초
- **로컬 학습 데이터:** `data/field_curriculum/compression_bench.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** channel split·가벼운 branch·concat·channel shuffle을 구현하고 FLOPs와 실제 latency를 분리해 기록한다.

**축소하거나 생략한 부분:** 원 논문은 ARM/GPU 최적화 라이브러리에서 실제 속도를 측정했다. 실습 latency는 현재 선택 장치의 상대 관찰값일 뿐이다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2: G1–G4 practical guidelines | Task 1 | 균형 channel과 memory access 직관 |
| §3, Figure 3(c): channel split | Task 2 | 반쪽 identity/branch |
| §3, Figure 3: concat + shuffle | Task 1–2 | 채널 permutation |
| §4: FLOPs vs measured speed | Task 3 | 선택 장치 latency를 별도 기록 |

## 미니 재현
FLOPs가 같아도 memory access·분기·element-wise 연산 때문에 속도가 다릅니다. 구조와 실제 시간을 따로 봅니다.

## 포트폴리오 해설: 정확도와 효율의 계약

$$Y=\operatorname{Shuffle}(X_1\VertP_{1\times1}D_{3\times3}P_{1\times1}(X_2))$$

- **기호 정의:** X1/X2는 channel split, D는 depthwise, ||는 concat, Shuffle은 channel permutation입니다.
- **수식의 역할:** 절반은 identity로 두고 계산 branch 뒤 채널을 섞어 정보 교환과 메모리 효율을 얻습니다.
- **구현 이유:** 정확도만 남기면 경량화의 비용 절감 주장을 검증할 수 없으므로,
  핵심 변환과 품질·비용 측정을 독립 함수와 method로 분리했습니다.
- **Task/코드 대응:** Task P의 논문 전용 class와 `forward`, `compute_loss`,
  `training_step`, `fit_portfolio_model`, `evaluate_portfolio_model`이
  §2: G1–G4 practical guidelines의 흐름을 명시합니다.
- **입출력 shape:** feature [B, C, H, W] → 두 [B, C/2, H, W] branch → shuffled output
- **평가:** parameter 수, accuracy, 실제 batch latency를 FLOPs와 별도 기록
- **원문 대비 한계:** 원 논문은 ARM/GPU 최적화 라이브러리에서 실제 속도를 측정했다. 실습 latency는 현재 선택 장치의 상대 관찰값일 뿐이다.

경량화 실험에서는 정확도만 보지 않고 parameter·sparsity·bit·latency 중 논문이
주장한 비용을 함께 기록해야 합니다. 아래 코드는 단계별 중간값을 남깁니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import math
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
torch.set_default_device(DEVICE)
print(ACCELERATOR.summary())


def locate(relative):
    for base in (Path.cwd(), *Path.cwd().parents):
        path = base / relative
        if path.exists():
            return path
    raise FileNotFoundError(relative)


dataset_path = locate("data/field_curriculum/compression_bench.npz")
raw = np.load(dataset_path, allow_pickle=False)
features = torch.from_numpy(raw["features"]).float().to(DEVICE)
labels = torch.from_numpy(raw["labels"]).long().to(DEVICE)
teacher_logits = torch.from_numpy(raw["teacher_logits"]).float().to(DEVICE)
teacher_weights = torch.from_numpy(raw["teacher_weights"]).float().to(DEVICE)
calibration = torch.from_numpy(raw["calibration_activations"]).float().to(DEVICE)
dense_weights = torch.from_numpy(raw["dense_weights"]).float().to(DEVICE)
train_idx = torch.from_numpy(raw["train_idx"]).long().to(DEVICE)
test_idx = torch.from_numpy(raw["test_idx"]).long().to(DEVICE)
images = torch.from_numpy(raw["images"]).float().to(DEVICE)
image_labels = torch.from_numpy(raw["image_labels"]).long().to(DEVICE)
image_train_idx = torch.from_numpy(raw["image_train_idx"]).long().to(DEVICE)
image_test_idx = torch.from_numpy(raw["image_test_idx"]).long().to(DEVICE)
token_embeddings = torch.from_numpy(raw["token_embeddings"]).float().to(DEVICE)
teacher_hidden = torch.from_numpy(raw["teacher_hidden"]).float().to(DEVICE)
teacher_attention = torch.from_numpy(raw["teacher_attention"]).float().to(DEVICE)
candidate_configs = torch.from_numpy(raw["candidate_configs"]).float()
assert features.shape == (512, 32) and teacher_logits.shape == (512, 4)
assert images.shape == (320, 1, 16, 16)
assert teacher_hidden.shape == (256, 6, 24)
print(
    dataset_path.name, "features", tuple(features.shape), "images", tuple(images.shape)
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2: G1–G4 practical guidelines / §3, Figure 3: concat + shuffle
- **이 셀의 책임:** Task 1 / Task 1–2
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 균형 channel과 memory access 직관 / 채널 permutation. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def channel_shuffle(x, groups=2):
    n, c, h, w = x.shape
    assert c % groups == 0
    return (
        x.reshape(n, groups, c // groups, h, w)
        .transpose(1, 2)
        .contiguous()
        .reshape(n, c, h, w)
    )


probe = torch.arange(8.0).view(1, 8, 1, 1)
shuffled = channel_shuffle(probe, 2)
assert shuffled.flatten().tolist() == [0, 4, 1, 5, 2, 6, 3, 7]

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3, Figure 3(c): channel split / §3, Figure 3: concat + shuffle
- **이 셀의 책임:** Task 2 / Task 1–2
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 반쪽 identity/branch / 채널 permutation. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class ShuffleUnit(nn.Module):
    def __init__(self, channels):
        super().__init__()
        half = channels // 2
        self.branch = nn.Sequential(
            nn.Conv2d(half, half, 1),
            nn.ReLU(),
            nn.Conv2d(half, half, 3, padding=1, groups=half),
            nn.Conv2d(half, half, 1),
            nn.ReLU(),
        )

    def forward(self, x):
        left, right = x.chunk(2, 1)
        return channel_shuffle(torch.cat([left, self.branch(right)], 1), 2)


unit = ShuffleUnit(8)
probe = torch.rand(2, 8, 8, 8)
assert unit(probe).shape == probe.shape and unit.branch[2].groups == 4

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §4: FLOPs vs measured speed
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 선택 장치 latency를 별도 기록. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class ShuffleMini(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Conv2d(1, 8, 3, padding=1)
        self.units = nn.Sequential(ShuffleUnit(8), ShuffleUnit(8))
        self.head = nn.Linear(8 * 8 * 8, 4)

    def forward(self, x):
        return self.head(F.max_pool2d(self.units(F.relu(self.stem(x))), 2).flatten(1))


model = ShuffleMini()
optimizer = torch.optim.Adam(model.parameters(), lr=0.025)
losses = []
for _ in range(35):
    optimizer.zero_grad()
    loss = F.cross_entropy(
        model(images[image_train_idx]), image_labels[image_train_idx]
    )
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
with torch.no_grad():
    for _ in range(10):
        model(images[:16])
    ACCELERATOR.synchronize()
    start = time.perf_counter()
    for _ in range(80):
        model(images[:16])
    ACCELERATOR.synchronize()
    milliseconds = (time.perf_counter() - start) * 1000 / 80
    accuracy = (
        (model(images[image_test_idx]).argmax(-1) == image_labels[image_test_idx])
        .float()
        .mean()
        .item()
    )
assert losses[-1] < losses[0] and accuracy > 0.65 and milliseconds > 0
plt.bar(
    ["parameters", "latency×100"],
    [sum(p.numel() for p in model.parameters()), milliseconds * 100],
)
plt.title("서로 다른 단위이므로 직접 비교 금지")
plt.show()
print(
    f"accuracy={accuracy:.1%}, local {DEVICE.type.upper()} "
    f"latency={milliseconds:.3f} ms/batch"
)

### Task P1 · 압축 대상과 핵심 모델

tensor shape와 비용 단위를 먼저 확인한 뒤 논문 전용 class의 `forward`와
`compute_loss`에 정확도·표현 보존 계약을 명시합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2: G1–G4 practical guidelines / §3, Figure 3(c): channel split / §3, Figure 3: concat + shuffle / §4: FLOPs vs measured speed
- **이 셀의 책임:** Task 1 / Task 2 / Task 1–2 / Task 3
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 균형 channel과 memory access 직관 / 반쪽 identity/branch / 채널 permutation / 선택 장치 latency를 별도 기록. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    channels: int = 8
    learning_rate: float = 0.02
    steps: int = 14


def prepare_batch() -> tuple[Tensor, Tensor]:
    return images[image_train_idx[:160]], image_labels[image_train_idx[:160]]


def channel_shuffle(inputs: Tensor, groups: int = 2) -> Tensor:
    batch, channels, height, width = inputs.shape
    grouped = inputs.reshape(batch, groups, channels // groups, height, width)
    shuffled = grouped.transpose(1, 2).contiguous()
    return shuffled.reshape(batch, channels, height, width)


class ShufflePortfolioBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        half = channels // 2
        self.branch = nn.Sequential(
            nn.Conv2d(half, half, 1),
            nn.ReLU(),
            nn.Conv2d(half, half, 3, padding=1, groups=half),
            nn.Conv2d(half, half, 1),
            nn.ReLU(),
        )

    def forward(self, inputs: Tensor) -> Tensor:
        identity, branch_input = inputs.chunk(2, dim=1)
        branch_output = self.branch(branch_input)
        return channel_shuffle(torch.cat([identity, branch_output], dim=1))


class ShufflenetV2PortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.stem = nn.Conv2d(1, config.channels, 3, padding=1)
        self.block = ShufflePortfolioBlock(config.channels)
        self.classifier = nn.Linear(config.channels, 4)

    def forward(self, inputs: Tensor) -> Tensor:
        hidden = F.relu(self.stem(inputs))
        hidden = self.block(hidden).mean(dim=(2, 3))
        return self.classifier(hidden)

    def compute_loss(self, logits: Tensor, targets: Tensor) -> Tensor:
        return F.cross_entropy(logits, targets)

    def training_step(self, inputs: Tensor, targets: Tensor) -> Tensor:
        return self.compute_loss(self(inputs), targets)

### Task P2 · 학습·pruning·distillation update

`fit_portfolio_model`에서 optimizer, mask/EMA/quantization 경계와 update 순서를
드러냅니다. 비용 절감이 학습 경로에 미치는 영향을 함께 읽으세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §3, Figure 3(c): channel split / §3, Figure 3: concat + shuffle
- **이 셀의 책임:** Task 2 / Task 1–2
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 반쪽 identity/branch / 채널 permutation. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: ShufflenetV2PortfolioModel) -> list[float]:
    inputs, targets = prepare_batch()
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(inputs, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 품질과 비용을 함께 평가

accuracy 또는 reconstruction 품질과 parameter·sparsity·bit·latency를 별도로
기록합니다. 로컬 지표는 원문 하드웨어 benchmark의 대체값이 아닙니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §4: FLOPs vs measured speed
- **이 셀의 책임:** Task 3
- **Tensor shape 계약:** feature map [B,C,H,W] → split/shuffle [B,C,H,W] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 선택 장치 latency를 별도 기록. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(model: ShufflenetV2PortfolioModel) -> dict[str, float]:
    benchmark_batch = images[image_test_idx[:32]]
    model.eval()
    with torch.no_grad():
        logits = model(images[image_test_idx])
        accuracy = (logits.argmax(1) == image_labels[image_test_idx]).float().mean()
        for _ in range(3):
            model(benchmark_batch)
        if benchmark_batch.is_cuda:
            torch.cuda.synchronize(benchmark_batch.device)
        started = time.perf_counter()
        for _ in range(25):
            model(benchmark_batch)
        if benchmark_batch.is_cuda:
            torch.cuda.synchronize(benchmark_batch.device)
        latency_ms = 1000.0 * (time.perf_counter() - started) / 25
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    depthwise_groups = model.block.branch[2].groups
    return {
        "test_accuracy": float(accuracy.cpu()),
        "parameter_count": float(parameter_count),
        "depthwise_groups": float(depthwise_groups),
        "measured_batch_latency_ms": float(latency_ms),
        "measured_images_per_second": float(32_000.0 / latency_ms),
    }


portfolio_model = ShufflenetV2PortfolioModel(PortfolioConfig())
portfolio_history = fit_portfolio_model(portfolio_model)
portfolio_metrics = evaluate_portfolio_model(portfolio_model)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 원 논문은 ARM/GPU 최적화 라이브러리에서 실제 속도를 측정했다. 실습 latency는 현재 선택 장치의 상대 관찰값일 뿐이다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.